# Experiment 50 - OOF Signal Search

Search compact feature/model recipes with leakage-safe 5-fold OOF ROC-AUC. No large submission library is created.

This version fixes the neighbourhood feature issue by using leave-one-out neighbourhood target rates for training rows and training-fold-only rates for validation rows.


In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5
TARGET = "Will_Buy_EV"
ROOT = Path("..")
TRAIN_PATH = ROOT / "data" / "train.csv"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(TRAIN_PATH)
if TARGET not in train.columns:
    raise ValueError(f"Missing target column: {TARGET}")
# The competition target is stored as Yes/No text, not 0/1 integers.
raw_target = train[TARGET]

if pd.api.types.is_numeric_dtype(raw_target):
    y = pd.to_numeric(raw_target, errors="raise").astype(np.int8).to_numpy()
else:
    target_text = raw_target.astype("string").str.strip().str.lower()
    target_map = {
        "yes": 1, "no": 0,
        "true": 1, "false": 0,
        "1": 1, "0": 0,
    }
    y = target_text.map(target_map)
    if y.isna().any():
        bad = sorted(target_text[y.isna()].dropna().unique().tolist())
        raise ValueError(f"Unexpected target values: {bad}")
    y = y.astype(np.int8).to_numpy()

if not np.isin(y, [0, 1]).all():
    raise ValueError("Target must contain only binary 0/1 values after conversion.")

print("Train shape:", train.shape)
print("Positive rate:", f"{y.mean():.6f}")


Train shape: (668665, 15)
Positive rate: 0.174645


## Deterministic features

Categoricals are encoded without target information. Numeric interactions, exact-value identity columns, and digit features are deterministic.


In [2]:
DROP_COLS = ["id", TARGET]
NUMERIC_COLS = ["Age","Annual_Income_USD","Daily_Commute_km","Number_of_Cars_Owned","Charging_Stations_Near_Home","Charging_Stations_Near_Work","Environmental_Concern_Level"]
CATEGORICAL_COLS = ["Gender","City_Type","Current_Car_Type","Home_Charging_Possible","Subsidy_Available","Range_Anxiety_Level"]
required = NUMERIC_COLS + CATEGORICAL_COLS
missing = [c for c in required if c not in train.columns]
if missing: raise ValueError(f"Missing required columns: {missing}")

df = train.drop(columns=DROP_COLS).copy()
for c in NUMERIC_COLS: df[c] = pd.to_numeric(df[c], errors="coerce")
for c in CATEGORICAL_COLS: df[c] = pd.Categorical(df[c]).codes.astype(np.int16)

df["Subsidy_x_EnvConcern"] = df["Subsidy_Available"] * df["Environmental_Concern_Level"]
df["Subsidy_x_Income"] = df["Subsidy_Available"] * np.log1p(np.clip(df["Annual_Income_USD"],0,None))
df["Subsidy_x_HomeCharging"] = df["Subsidy_Available"] * df["Home_Charging_Possible"]
df["Income_per_Commute"] = df["Annual_Income_USD"] / (df["Daily_Commute_km"] + 1.0)
df["Income_per_Age"] = df["Annual_Income_USD"] / (df["Age"] + 1.0)
df["Commute_x_Cars"] = df["Daily_Commute_km"] * (df["Number_of_Cars_Owned"] + 1.0)

IDENTITY_COLS = ["Age","Annual_Income_USD","Daily_Commute_km","Charging_Stations_Near_Home","Charging_Stations_Near_Work"]
for c in IDENTITY_COLS: df[f"{c}__value"] = df[c].astype("string").fillna("__NA__")
for c in ["Age","Annual_Income_USD","Daily_Commute_km"]:
    vals = pd.to_numeric(df[c], errors="coerce").fillna(-999).astype(np.int64)
    df[f"{c}__last_digit"] = np.abs(vals) % 10
    df[f"{c}__mod5"] = np.abs(vals) % 5
    df[f"{c}__mod10"] = np.abs(vals) % 10

BASE_FEATURES = [c for c in df.columns if not c.endswith("__value")]
print("Base feature count:", len(BASE_FEATURES))


Base feature count: 28


## Leakage-safe target encoding

Every target encoding map is fitted separately inside each fold using training labels only. Unknown values use the training-fold global mean.


In [3]:
TE_SOURCE_COLS = ["Age__value","Annual_Income_USD__value","Daily_Commute_km__value","Charging_Stations_Near_Home__value","Charging_Stations_Near_Work__value","City_Type","Current_Car_Type","Subsidy_Available","Home_Charging_Possible"]

def fit_te_maps(X_tr, y_tr, cols, alpha=30.0):
    gm = float(np.mean(y_tr)); maps = {}
    for c in cols:
        keys = X_tr[c].astype("string").fillna("__NA__")
        stats = pd.DataFrame({"key":keys.to_numpy(),"y":y_tr}).groupby("key",sort=False)["y"].agg(["sum","count"])
        stats["rate"] = (stats["sum"] + alpha*gm) / (stats["count"] + alpha)
        maps[c] = (stats["rate"].to_dict(), gm)
    return maps

def apply_te_maps(X, maps):
    out = pd.DataFrame(index=X.index)
    for c,(mapping,gm) in maps.items():
        out[f"TE__{c}"] = X[c].astype("string").fillna("__NA__").map(mapping).fillna(gm).astype(np.float32)
    return out


## Leakage-safe neighbourhood rates

The reference set is always the training fold. For training rows, the row's own target is removed from each neighbourhood sum.


In [4]:
NEIGHBOURHOOD_SPECS = [("Annual_Income_USD",5000),("Annual_Income_USD",10000),("Daily_Commute_km",2),("Daily_Commute_km",5),("Age",2),("Age",5)]

def fit_neighbourhood_maps(X_tr,y_tr):
    gm=float(np.mean(y_tr)); maps={}
    for col,window in NEIGHBOURHOOD_SPECS:
        values=pd.to_numeric(X_tr[col],errors="coerce").to_numpy(dtype=np.float64)
        finite=np.isfinite(values)
        xs=values[finite]; ys=y_tr[finite].astype(np.float64)
        order=np.argsort(xs,kind="mergesort")
        xs=xs[order]; ys=ys[order]
        maps[(col,window)] = (xs,np.concatenate(([0.0],np.cumsum(ys))),gm)
    return maps

def apply_neighbourhood_maps(X,maps,y_for_training=None):
    out=pd.DataFrame(index=X.index)
    for (col,window),(xs,prefix,gm) in maps.items():
        values=pd.to_numeric(X[col],errors="coerce").to_numpy(dtype=np.float64)
        finite=np.isfinite(values); sums=np.zeros(len(values)); counts=np.zeros(len(values),dtype=np.int64)
        if finite.any():
            v=values[finite]
            left=np.searchsorted(xs,v-window,side="left"); right=np.searchsorted(xs,v+window,side="right")
            sums[finite]=prefix[right]-prefix[left]; counts[finite]=right-left
        if y_for_training is not None:
            if len(y_for_training)!=len(values): raise ValueError("Training target length mismatch")
            sums[finite]-=y_for_training[finite].astype(np.float64); counts[finite]-=1
        counts=np.maximum(counts,0)
        rates=(sums+20.0*gm)/(counts+20.0); rates[~finite]=gm
        out[f"NB__{col}_{window}"]=rates.astype(np.float32)
    return out


## Model search

Four XGBoost configurations are tested across three feature modes. No test labels are used.


In [5]:
CONFIGS={
"A_deep_regularized":dict(max_depth=6,min_child_weight=2,subsample=.90,colsample_bytree=.90,learning_rate=.035,n_estimators=1800,reg_lambda=3.0,gamma=0.0),
"B_shallow_high_subsample":dict(max_depth=5,min_child_weight=2,subsample=.98,colsample_bytree=.95,learning_rate=.035,n_estimators=1800,reg_lambda=3.0,gamma=0.0),
"C_depth7_regularized":dict(max_depth=7,min_child_weight=3,subsample=.90,colsample_bytree=.90,learning_rate=.030,n_estimators=2000,reg_lambda=4.0,gamma=0.0),
"D_more_regularization":dict(max_depth=6,min_child_weight=4,subsample=.92,colsample_bytree=.92,learning_rate=.035,n_estimators=1800,reg_lambda=6.0,gamma=.05)}
MODES=["base","target_encoded","target_plus_neighbourhood"]
skf=StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)
X_all=df.reset_index(drop=True); model_oof={}; summary_rows=[]

for mode in MODES:
    for cfg_name,cfg in CONFIGS.items():
        name=f"{mode}__{cfg_name}"; oof=np.zeros(len(X_all),dtype=np.float64); fold_scores=[]
        for fold,(tr_idx,va_idx) in enumerate(skf.split(X_all,y),1):
            Xtr=X_all.iloc[tr_idx].copy(); Xva=X_all.iloc[va_idx].copy(); ytr=y[tr_idx]; yva=y[va_idx]
            parts_tr=[Xtr[BASE_FEATURES].reset_index(drop=True)]; parts_va=[Xva[BASE_FEATURES].reset_index(drop=True)]
            if mode in ("target_encoded","target_plus_neighbourhood"):
                maps=fit_te_maps(Xtr,ytr,TE_SOURCE_COLS); parts_tr.append(apply_te_maps(Xtr,maps).reset_index(drop=True)); parts_va.append(apply_te_maps(Xva,maps).reset_index(drop=True))
            if mode=="target_plus_neighbourhood":
                nb=fit_neighbourhood_maps(Xtr,ytr); parts_tr.append(apply_neighbourhood_maps(Xtr,nb,y_for_training=ytr).reset_index(drop=True)); parts_va.append(apply_neighbourhood_maps(Xva,nb).reset_index(drop=True))
            Xtr2=pd.concat(parts_tr,axis=1).replace([np.inf,-np.inf],np.nan).fillna(0).astype(np.float32)
            Xva2=pd.concat(parts_va,axis=1).replace([np.inf,-np.inf],np.nan).fillna(0).astype(np.float32)
            model=XGBClassifier(objective="binary:logistic",eval_metric="auc",tree_method="hist",random_state=SEED,n_jobs=-1,verbosity=0,**cfg)
            model.fit(Xtr2,ytr,eval_set=[(Xva2,yva)],verbose=False)
            pred=model.predict_proba(Xva2)[:,1]; oof[va_idx]=pred
            auc=float(roc_auc_score(yva,pred)); fold_scores.append(auc); print(f"{name} | fold {fold}: {auc:.6f}")
        overall=float(roc_auc_score(y,oof)); model_oof[name]=oof
        summary_rows.append({"candidate":name,"type":"single_model","oof_auc":overall,"fold_mean":float(np.mean(fold_scores)),"fold_std":float(np.std(fold_scores)),"fold_scores":json.dumps([round(v,7) for v in fold_scores])})
        print(f"{name} | OOF: {overall:.6f}\n")

single_df=pd.DataFrame(summary_rows).sort_values("oof_auc",ascending=False).reset_index(drop=True)
display(single_df[["candidate","oof_auc","fold_mean","fold_std"]])


base__A_deep_regularized | fold 1: 0.940806
base__A_deep_regularized | fold 2: 0.941861
base__A_deep_regularized | fold 3: 0.942953
base__A_deep_regularized | fold 4: 0.942526
base__A_deep_regularized | fold 5: 0.941964
base__A_deep_regularized | OOF: 0.942018

base__B_shallow_high_subsample | fold 1: 0.941095
base__B_shallow_high_subsample | fold 2: 0.942161
base__B_shallow_high_subsample | fold 3: 0.943253
base__B_shallow_high_subsample | fold 4: 0.942830
base__B_shallow_high_subsample | fold 5: 0.942147
base__B_shallow_high_subsample | OOF: 0.942293

base__C_depth7_regularized | fold 1: 0.940495
base__C_depth7_regularized | fold 2: 0.941480
base__C_depth7_regularized | fold 3: 0.942546
base__C_depth7_regularized | fold 4: 0.942011
base__C_depth7_regularized | fold 5: 0.941478
base__C_depth7_regularized | OOF: 0.941597

base__D_more_regularization | fold 1: 0.940820
base__D_more_regularization | fold 2: 0.941821
base__D_more_regularization | fold 3: 0.943006
base__D_more_regularizati

,candidate,oof_auc,fold_mean,fold_std
0,target_encoded__B_shallow_high_subsample,0.943560,0.943568,0.000418
1,target_encoded__D_more_regularization,0.943134,0.943141,0.000470
2,target_encoded__A_deep_regularized,0.943108,0.943114,0.000485
3,target_encoded__C_depth7_regularized,0.942683,0.942689,0.000476
4,base__B_shallow_high_subsample,0.942293,0.942297,0.000733
5,base__D_more_regularization,0.942019,0.942024,0.000739
6,base__A_deep_regularized,0.942018,0.942022,0.000725
7,base__C_depth7_regularized,0.941597,0.941602,0.000680
8,target_plus_neighbourhood__D_more_regularization,0.787989,0.790912,0.040683
9,target_plus_neighbourhood__B_shallow_high_subs...,0.772656,0.777050,0.031742


## In-memory rank blend search

Blend candidates are evaluated against OOF labels but are not written as prediction CSVs.


In [6]:
def rank01(a): return pd.Series(a).rank(method="average").to_numpy(dtype=np.float64)/len(a)
rank_cache={}; top_names=single_df.head(8)["candidate"].tolist()
for n in top_names: rank_cache[n]=rank01(model_oof[n])
blend_rows=[]

for i in range(len(top_names)):
    for j in range(i+1,len(top_names)):
        a,b=top_names[i],top_names[j]
        for w in (.25,.50,.75):
            pred=w*rank_cache[a]+(1-w)*rank_cache[b]
            blend_rows.append({"candidate":f"rank2__{a}__{b}__{w:.2f}","type":"two_way_rank_blend","oof_auc":float(roc_auc_score(y,pred)),"fold_mean":np.nan,"fold_std":np.nan,"fold_scores":""})

top5=top_names[:5]
for i in range(len(top5)):
    for j in range(i+1,len(top5)):
        for k in range(j+1,len(top5)):
            a,b,c=top5[i],top5[j],top5[k]
            for wa,wb,wc in ((.50,.25,.25),(.60,.20,.20),(.70,.15,.15),(.40,.30,.30)):
                pred=wa*rank_cache[a]+wb*rank_cache[b]+wc*rank_cache[c]
                blend_rows.append({"candidate":f"rank3__{a}__{b}__{c}__{wa:.2f}_{wb:.2f}_{wc:.2f}","type":"three_way_rank_blend","oof_auc":float(roc_auc_score(y,pred)),"fold_mean":np.nan,"fold_std":np.nan,"fold_scores":""})

blend_df=pd.DataFrame(blend_rows)
all_results=pd.concat([single_df,blend_df],ignore_index=True).sort_values("oof_auc",ascending=False).reset_index(drop=True)
display(all_results.head(20)[["candidate","type","oof_auc"]])


,candidate,type,oof_auc
0,rank2__target_encoded__B_shallow_high_subsampl...,two_way_rank_blend,0.944402
1,rank3__target_encoded__B_shallow_high_subsampl...,three_way_rank_blend,0.944341
2,rank2__target_encoded__B_shallow_high_subsampl...,two_way_rank_blend,0.944341
3,rank3__target_encoded__B_shallow_high_subsampl...,three_way_rank_blend,0.944339
4,rank3__target_encoded__B_shallow_high_subsampl...,three_way_rank_blend,0.944289
5,rank2__target_encoded__B_shallow_high_subsampl...,two_way_rank_blend,0.944288
6,rank3__target_encoded__B_shallow_high_subsampl...,three_way_rank_blend,0.944287
7,rank2__target_encoded__B_shallow_high_subsampl...,two_way_rank_blend,0.944285
8,rank2__target_encoded__B_shallow_high_subsampl...,two_way_rank_blend,0.944283
9,rank2__target_encoded__B_shallow_high_subsampl...,two_way_rank_blend,0.944278


In [7]:
search_path=RESULTS_DIR/"exp50_search_results.csv"
recipe_path=RESULTS_DIR/"exp50_best_recipe.json"
all_results.to_csv(search_path,index=False)
best=all_results.iloc[0]
recipe={"experiment":50,"metric":"ROC-AUC","best_candidate":str(best["candidate"]),"best_oof_auc":float(best["oof_auc"]),"candidate_type":str(best["type"]),"note":"OOF score only. Test labels are unavailable locally. No submission CSV was generated by Experiment 50."}
with open(recipe_path,"w",encoding="utf-8") as f: json.dump(recipe,f,indent=2)
print("Saved:",search_path); print("Saved:",recipe_path); print("Best:",recipe["best_candidate"],f"{recipe['best_oof_auc']:.6f}")


Saved: ..\results\exp50_search_results.csv
Saved: ..\results\exp50_best_recipe.json
Best: rank2__target_encoded__B_shallow_high_subsample__base__B_shallow_high_subsample__0.50 0.944402


## Final integrity checks

Every model must have exactly one finite OOF prediction per training row.


In [8]:
for name,pred in model_oof.items():
    assert len(pred)==len(y),f"Length mismatch: {name}"
    assert np.all(np.isfinite(pred)),f"Non-finite prediction: {name}"
    assert np.count_nonzero(pred)==len(y),f"Missing OOF predictions: {name}"
assert len(all_results)>0
assert np.all(np.isfinite(all_results["oof_auc"].to_numpy(dtype=float)))
print("All Exp50 checks passed.")
print("Models tested:",len(model_oof))
print("In-memory blends tested:",len(blend_df))


All Exp50 checks passed.
Models tested: 12
In-memory blends tested: 124
